Import Python modules

In [1]:
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from collections import defaultdict
import glob
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(font_scale=1.0, style='ticks', palette='colorblind')

Read in data

In [ ]:
# Input variables
subtype = 'H7N9'
output_dir = os.path.join('../results')

In [7]:
print(f"Processing data for {subtype}")

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

Processing data for H7N9


In [ ]:
# Read in sequence data and metadata
data_dir = os.path.join('../data', subtype)
fasta_files = glob.glob(os.path.join(data_dir, "*.fasta"))
metadata_files = glob.glob(os.path.join(data_dir, "*.xls"))
assert len(fasta_files) == len(metadata_files) == 1, "There should be exactly one fasta and one metadata file."

In [5]:
# Iterate over sequences, group by segment, and make one FASTA file per
# segment with all sequences for that segment
segment_records = defaultdict(list)
epi_ids = set()
epi_isl_ids = set()
for fasta_file in fasta_files:
    
    # Read in records
    records = list(SeqIO.parse(fasta_file, "fasta"))
    print(f"Read {len(records)} records from {fasta_file}")
    
    # Iterate through records and parse metadata from the sequence ID
    for record in records:
        
        # Parse the sequence ID
        try:
            (epi, segment, name, epi_isl, seq_subtype) = record.id.split('|')
            
            # Skip record if we've already seen this epi ID
            if epi in epi_ids:
                print(f"Warning: Duplicate EPI ID {epi} found. Skipping record.")
                continue
            epi_ids.add(epi)

            # Check that the subtype matches what we expect
            if not seq_subtype.endswith(subtype):
                print(f"Warning: Record {record.id} has mismatched subtype: {seq_subtype} vs expected {subtype}")
                continue
            
            # Update record ID and description to just use the EPI_ISL
            record.id = epi_isl
            record.description = epi_isl
            
            # Store record by segment
            segment_records[segment].append(record)

            # Update set of EPI_ISL IDs
            epi_isl_ids.add(epi_isl)
            
        except ValueError:
            print(f"Warning: Could not parse ID for record: {record.id}")
            continue

# For each segment, print summary and then write the records to a FASTA file
print(f"\nTotal unique EPI_ISL IDs: {len(epi_isl_ids)}")
print("\nSummary of records by segment:")
for segment, records in segment_records.items():
    segment_output_dir = os.path.join(output_dir, f'{subtype}-{segment}/')
    if not os.path.isdir(segment_output_dir):
        os.makedirs(segment_output_dir)
    output_file = os.path.join(segment_output_dir, f"raw_sequences.fasta")
    SeqIO.write(records, output_file, "fasta")
    print(f"Segment {segment}: {len(records)} records written to {output_file}")

Read 22194 records from ../data/H7N9/H7N9.fasta

Total unique EPI_ISL IDs: 3216

Summary of records by segment:
Segment NA: 3018 records written to ../results/H7N9-NA/raw_sequences.fasta
Segment NS: 2700 records written to ../results/H7N9-NS/raw_sequences.fasta
Segment HA: 3069 records written to ../results/H7N9-HA/raw_sequences.fasta
Segment MP: 2702 records written to ../results/H7N9-MP/raw_sequences.fasta
Segment NP: 2677 records written to ../results/H7N9-NP/raw_sequences.fasta
Segment PB1: 2680 records written to ../results/H7N9-PB1/raw_sequences.fasta
Segment PB2: 2684 records written to ../results/H7N9-PB2/raw_sequences.fasta
Segment PA: 2664 records written to ../results/H7N9-PA/raw_sequences.fasta


In [ ]:
# Load all metadata and write to an output CSV file
dfs = []
for metadata_file in metadata_files:

    # Read in data from the first sheet
    df = pd.read_excel(metadata_file, sheet_name=0)
    print(f"Read {len(df)} records from {metadata_file}")
    dfs.append(df)

# Concatenate all metadata dataframes
metadata_df = pd.concat(dfs)

# Make sure that all Isolate_Id values are unique
assert sum(metadata_df.duplicated(subset=['Isolate_Id'])) == 0, "Duplicate Isolate_Id values found in metadata."
assert len(metadata_df) == len(epi_isl_ids), "Number of records in metadata does not match number of unique EPI_ISL IDs."

# Write metadata to CSV
metadata_output_file = os.path.join(output_dir, f"{subtype}_metadata.csv")
print(f"\nWriting metadata to {metadata_output_file}")
metadata_df.to_csv(metadata_output_file, index=False)


Read 3216 records from ../data/H7N9/H7N9.xls

Writing metadata to ../results/H7N9_metadata.csv
